# PND 2025: dos microdados ao Power BI

Demonstracao do ciclo separado da Prova Nacional Docente de 2025. Diferentemente das series historicas agregadas, cada linha valida da fato representa um registro individual da prova. A saida e `data/gold/fatos/fato_pnd.parquet`.

## Ciclo

Microdados, dicionario e documentacao oficial -> auditoria da populacao e variaveis -> Bronze -> Silver -> fato Gold com padrao de desempenho -> Power BI. A classificacao usa `NT_OBJ`: abaixo de 50, nao proficiente; de 50 a menos de 70, Padrao 1; a partir de 70, Padrao 2.

## Criterios de tratamento e populacao analitica

A Bronze preserva os 1.087.359 registros e as colunas fisicas do arquivo. A Silver seleciona `TP_PRES = 555` e exige preenchimento simultaneo de `PROFICIENCIA`, `NT_OBJ`, `NT_DIS`, `NT_GER` e `QT_ACERTOS`. Isso produz 759.140 registros: 966 presentes sem resultados completos ficam fora e 12 registros com `TP_PRES = 888`, embora completos, tambem ficam fora da populacao definida.

Os codigos sao convertidos para tipos consistentes, `CO_GRUPO` e associado a uma das 17 areas oficiais, e UF e municipio referem-se ao local da prova. Arquivo, linha e granularidade de origem permanecem na Silver para rastreabilidade. Valores negativos de `PROFICIENCIA` sao validos na escala TRI e nao sao truncados.

A Gold seleciona os nove campos de consumo e acrescenta `PADRAO_DESEMPENHO`, derivado exclusivamente de `NT_OBJ`: menor que 50, `NAO_PROFICIENTE`; de 50 a menos de 70, `PADRAO_1`; a partir de 70, `PADRAO_2`. `NT_GER` e `NT_DIS` nao sao usados como cortes de proficiencia.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
DIRETORIO_ATUAL = Path.cwd().resolve()
ROOT = next((p for p in (DIRETORIO_ATUAL, *DIRETORIO_ATUAL.parents) if (p / 'src' / 'pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')
print(f'Raiz do projeto: {ROOT}')
EXECUTAR_AUDITORIAS = False
EXECUTAR_PIPELINE_INDICADOR = False
pd.set_option('display.max_columns', 30)

In [ ]:
AUDITORIAS = [
 'src/auditoria_pnd/inspecionar_pnd.py',
 'src/auditoria_pnd/inspecionar_dicionario_pnd.py',
 'src/auditoria_pnd/extrair_variaveis_chave_pnd.py',
 'src/auditoria_pnd/auditar_populacao_pnd.py',
 'src/auditoria_pnd/reproduzir_indicadores_pnd.py',
 'src/auditoria_pnd/validar_resultados_pnd.py',
]
ETAPAS = [
 'src/bronze/pnd/ingest_pnd.py',
 'src/bronze/pnd/validar_bronze_pnd.py',
 'src/silver/pnd/transformar_pnd.py',
 'src/silver/pnd/validar_silver_pnd.py',
 'src/gold/pnd/transformar_pnd.py',
 'src/gold/pnd/validar_fato_pnd.py',
]
def executar(scripts):
    for script in scripts:
        print(f'\n>>> {script}')
        subprocess.run([sys.executable, script], cwd=ROOT, check=True)
print('Auditorias:', *AUDITORIAS, sep='\n- ')
print('\nEtapas:', *ETAPAS, sep='\n- ')

In [ ]:
if EXECUTAR_AUDITORIAS:
    executar(AUDITORIAS)
else:
    print('Auditorias nao executadas.')
if EXECUTAR_PIPELINE_INDICADOR:
    executar(ETAPAS)
else:
    print('Pipeline nao executado; leitura dos artefatos existentes.')

## Evidencias por camada

A Raw e inventariada sem exibir dados individuais. As amostras das camadas derivadas devem ser usadas com cuidado na apresentacao, ainda que o conjunto publicado nao contenha identificador individual direto.

In [ ]:
raw = sorted((ROOT / 'data/raw/pnd').glob('*'))
bronze_path = ROOT / 'data/bronze/pnd/pnd_2025.parquet'
silver_path = ROOT / 'data/silver/pnd/pnd_2025.parquet'
gold_path = ROOT / 'data/gold/fatos/fato_pnd.parquet'
pd.DataFrame({'camada':['Raw','Bronze','Silver','Gold'], 'arquivos':[len(raw),int(bronze_path.exists()),int(silver_path.exists()),int(gold_path.exists())], 'destino':[str(ROOT/'data/raw/pnd'),str(bronze_path),str(silver_path),str(gold_path)]})

In [ ]:
bronze = pd.read_parquet(bronze_path)
silver = pd.read_parquet(silver_path)
gold = pd.read_parquet(gold_path)
display(bronze.head(3), silver.head(3), gold.head(3))
pd.DataFrame({'camada':['Bronze','Silver','Gold'], 'linhas':[len(bronze),len(silver),len(gold)], 'colunas':[len(bronze.columns),len(silver.columns),len(gold.columns)]})

### Evidencia da selecao da populacao

O diagnostico abaixo reproduz os criterios de presenca e completude diretamente da Bronze. A comparacao de esquemas mostra quais campos de auditoria ficam na Silver e qual classificacao e acrescentada na Gold.

In [ ]:
dados = bronze.loc[bronze['_linha_origem'] > 1].copy()
campos_resultado = ['col_013','col_014','col_015','col_016','col_017']
def preenchido(serie):
    texto = serie.astype('string').str.strip()
    return texto.notna() & texto.ne('') & texto.str.upper().ne('NA')
resultados_completos = pd.concat([preenchido(dados[c]) for c in campos_resultado], axis=1).all(axis=1)
tp_pres = pd.to_numeric(dados['col_011'], errors='coerce')
diagnostico = pd.DataFrame({'criterio':['Registros da fonte','Cinco resultados completos','TP_PRES=555 e resultados completos','TP_PRES=555 sem resultados completos','TP_PRES=888 e resultados completos'], 'registros':[len(dados),int(resultados_completos.sum()),int((tp_pres.eq(555)&resultados_completos).sum()),int((tp_pres.eq(555)&~resultados_completos).sum()),int((tp_pres.eq(888)&resultados_completos).sum())]})
display(diagnostico)
display(silver[['ANO','UF_PROVA','CO_MUNICIPIO_PROVA','CO_GRUPO','PROFICIENCIA','NT_OBJ','NT_DIS','NT_GER','QT_ACERTOS','ARQUIVO_ORIGEM','LINHA_ORIGEM_BRONZE','GRANULARIDADE_ORIGEM']].head(4))
pd.DataFrame({'Silver_preserva_para_auditoria':[', '.join(c for c in silver.columns if c not in gold.columns)], 'Gold_acrescenta':[', '.join(c for c in gold.columns if c not in silver.columns)], 'Gold_entrega_ao_BI':[', '.join(gold.columns)]})

In [ ]:
distribuicao = gold['PADRAO_DESEMPENHO'].value_counts().rename_axis('PADRAO').to_frame('PARTICIPANTES')
distribuicao['PERCENTUAL'] = 100 * distribuicao['PARTICIPANTES'] / len(gold)
display(distribuicao)
ax = distribuicao['PERCENTUAL'].plot(kind='bar', figsize=(8,5), color=['#425466','#d08a3c','#2f7d68'])
ax.set(title='Distribuicao dos participantes por padrao', xlabel='Padrao de desempenho', ylabel='Participantes (%)')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
plt.show()

## Disponibilizacao para o BI

A PND utiliza `DIM_UF`, `DIM_TEMPO`, `DIM_AREA_PND` e `DIM_MUNICIPIO`. Contagens, medias e percentuais sao calculados pelas medidas DAX conforme o filtro do relatorio. O notebook nao publica dados novos nem substitui a validacao global do modelo.